### Propagar __Capacidad__ y __Velocidad__ para llenar los gaps en las avenidas que no hicieron match con el codigo de Daniel

In [2]:
import pandas as pandas
import geopandas as gpd
import numpy as np
import os

In [3]:
folder_matched = r'/Users/jeannettearjona/Library/CloudStorage/OneDrive-InstitutoTecnologicoydeEstudiosSuperioresdeMonterrey/Modelación Urbana - Red Vial Guadalajara/VisumLinks With TransCAD Atts'
analysis_path = os.path.join(folder_matched, "Analysis")
shp_path = os.path.join(folder_matched, "links con attributos x nombre")
shp_path2 = os.path.join(shp_path, "without highways")

### Pre-procesamiento de matched links

In [4]:
links = gpd.read_file(os.path.join(folder_matched, "edges_osnmx_con_velocidades_y_capacidad2.shp"))

#Cols
nombre = "name"
capacidad = "CAPACIDAD"
vel_prom = "Velocidad_"
v0 = "Limite_vel"

# 0) Limpiar atributos mal heredados por tipo de vía
highways_invalidos = [
    "residential",
    "cycleway",
    "footway",
    "pedestrian",
    "path",
    "steps",
]

mask_invalidos = (
    links["highway"].isna() | #if highway = NULL es linea del metro y no deberia tener atts heredados
    links['highway']
    .astype(str)
    .str.contains("|".join(highways_invalidos), case=False, na=False)
)

print("Links con atributos antes de limpiar:")
print(
    (
        links[capacidad].notna() &
        links[vel_prom].notna() &
        links[v0].notna()
    ).sum()
)

print("Links con atributos eliminados por highway inválido:")
print(
    (
        mask_invalidos &
        links[capacidad].notna() &
        links[vel_prom].notna() &
        links[v0].notna()
    ).sum()
)

# 1) Change to NaN the attributes of links which highway type shouldn't have gotten attributes
# Ex. Living streets or cycleways that got the attributes of a main avenue (wrong!)
links.loc[mask_invalidos, [capacidad, vel_prom, v0]] = np.nan

Links con atributos antes de limpiar:
12083
Links con atributos eliminados por highway inválido:
4022


### Links con capacidades y velocidades de TransCAD (los que pudo matchear Daniel)

In [5]:
#links con nombre
links_with_name = links[links[nombre].notnull()].copy()

#links with NOMBRE, capacidad y velocidad
links_con_atts = links_with_name[
    links_with_name[capacidad].notna() &
    links_with_name[vel_prom].notna() &
    links_with_name[v0].notna()
].copy()

#links with NOMBRE pero sin capacidad y velocidad
links_sin_atts = links_with_name[
    links_with_name[capacidad].isna() &
    links_with_name[vel_prom].isna() &
    links_with_name[v0].isna()
].copy()

#ALL links con capacidad y velocidad
all_links_con_atts = links[
    links[capacidad].notna() &
    links[vel_prom].notna() &
    links[v0].notna()
].copy()

print(f"OSMNX links previo a Visum import: {len(links)}")
print(f"Todos los links (con o sin nombre) con CAPACIDAD y VELOCIDAD: {len(all_links_con_atts)}")
#print(f"Links con nombre: {len(links_with_name)}")
print(f"Links con nombre, CAPACIDAD y VELOCIDAD: {len(links_con_atts)}")
#print(f"Links sin nombre, CAPACIDAD y VELOCIDAD: {len(links_sin_atts)}")


OSMNX links previo a Visum import: 494896
Todos los links (con o sin nombre) con CAPACIDAD y VELOCIDAD: 8061
Links con nombre, CAPACIDAD y VELOCIDAD: 7110


In [7]:
# Links con atributos (con y sin nombre) 8061
all_links_con_atts['highway'].value_counts(dropna=False)

highway
primary                          2875
secondary                        2097
tertiary                         1216
trunk                             569
unclassified                      265
service                           215
motorway                          187
primary_link                      159
busway                            158
living_street                      84
trunk_link                         80
motorway_link                      78
secondary_link                     38
tertiary_link                      23
track                              10
['service', 'track']                2
['busway', 'service']               2
['tertiary', 'trunk_link']          1
['tertiary_link', 'tertiary']       1
['primary', 'trunk']                1
Name: count, dtype: int64

In [8]:
# Links con atributos y nombre 7110
links_con_atts['highway'].value_counts(dropna=False)

highway
primary                       2827
secondary                     2019
tertiary                      1122
trunk                          564
motorway                       186
busway                         154
unclassified                   125
living_street                   58
service                         20
motorway_link                    9
trunk_link                       8
track                            8
primary_link                     4
secondary_link                   2
['busway', 'service']            2
['tertiary', 'trunk_link']       1
['primary', 'trunk']             1
Name: count, dtype: int64

In [9]:
#cuantos nombres de los links_sin_atts existen en links_con_atts
nombres_con_atts = set(links_con_atts[nombre])
nombres_sin_atts = set(links_sin_atts[nombre])

#links sin atts que cuyo nombre existe en links con atts
links_rellenables_por_nombre = links_sin_atts[
    links_sin_atts[nombre].isin(nombres_con_atts)
].copy()

nombres_match = nombres_sin_atts.intersection(nombres_con_atts)

print(f"{len(nombres_match)} nombres únicos pueden servir para propagación")
print(f"Esos {len(nombres_match)} cubren {len(links_rellenables_por_nombre)} links sin attributos")


270 nombres únicos pueden servir para propagación
Esos 270 cubren 25486 links sin attributos


### ¿Todos los que tienen el mismo nombre tienen __una sola misma__ capacidad y velocidad?

In [10]:
revision_nombres = (
    links_con_atts
    .groupby(nombre)
    .agg(
        n_cap=(capacidad, "nunique"),
        n_vel=(vel_prom, "nunique"),
        n_v0=(v0, "nunique")
    )
    .reset_index()
)
nombres_inconsistentes = revision_nombres[
    (revision_nombres["n_cap"] > 1) |
    (revision_nombres["n_vel"] > 1) |
    (revision_nombres["n_v0"] > 1)
]

revision_nombres.to_excel(os.path.join(analysis_path,"Revision_porNombres.xlsx"), index=False)
print(f"{len(nombres_inconsistentes)} links diferentes valores de capacidad y velocidad para el mismo nombre")

224 links diferentes valores de capacidad y velocidad para el mismo nombre


#### ¿qué tanta diferencia hay en capacidades y velocidades dentro de algún nombre que presente variaciones y a qué se deben?
- no son confiables porque vienen de una red con geometrías excesivamente agregadas entonces esos cambios de capacidad es porque un solo link pasa de representar a 2 vias a representar a 4 o 6

In [11]:
#Case study
nombre_prueba = "Avenida Doctor Roberto Michel"

links_con_atts.loc[
    links_con_atts[nombre] == nombre_prueba,
    [nombre, capacidad, vel_prom, v0]
].drop_duplicates()

,name,CAPACIDAD,Velocidad_,Limite_vel
101665,Avenida Doctor Roberto Michel,12000.0,26.000000,40.0
101666,Avenida Doctor Roberto Michel,12000.0,26.000007,40.0
101675,Avenida Doctor Roberto Michel,12000.0,25.999950,40.0
101679,Avenida Doctor Roberto Michel,12000.0,26.000057,40.0
101688,Avenida Doctor Roberto Michel,12000.0,26.000079,40.0
101701,Avenida Doctor Roberto Michel,8000.0,26.000047,40.0
102023,Avenida Doctor Roberto Michel,9000.0,21.999453,40.0
105487,Avenida Doctor Roberto Michel,9000.0,22.000052,40.0
105530,Avenida Doctor Roberto Michel,6000.0,21.999988,40.0
105534,Avenida Doctor Roberto Michel,6000.0,22.000181,40.0


### __usar la media (o mediana) por nombre__ para capacidad, velocidad, v0

In [12]:
#"Para cada avenida voy a usar el valor típico que ya existe en esa avenida."

#1) Agrupar por nombre usando la MEDIANA de capacidad, vel_prom, v0
atts_por_nombre = (
    links_con_atts
    .groupby(nombre)[[capacidad, vel_prom, v0]]
    .median()
    .reset_index()
    .rename(columns={
        capacidad: "cap_med",
        vel_prom: "velprom_med",
        v0: "limvel_med"
    })
)

#2) Merge a todos los links
links_fill = links.merge(atts_por_nombre, on=nombre, how='left')

In [13]:
atts_por_nombre[atts_por_nombre[nombre]=='Avenida Doctor Roberto Michel']

,name,cap_med,velprom_med,limvel_med
42,Avenida Doctor Roberto Michel,9000.0,25.99995,40.0


In [ ]:
# columnas finales empiezan vacías
links_fill["cap_final"] = np.nan
links_fill["velprom_final"] = np.nan
links_fill["limvel_final"] = np.nan

highways_validos_para_recibir = [
    "primary",
    "secondary",
    "tertiary",
    "trunk",
    "unclassified",
    "service",
    "motorway",
    "motorway_link",
    "trunk_link",
    "primary_link",
    "secondary_link",
    "tertiary_link"
]

#solo los links con highway valido pueden recibir atributos
mask_recibir_atts = (
    links_fill["highway"]
    .astype(str)
    .str.contains("|".join(highways_validos_para_recibir), case=False, na=False)
)

# 1) Links con nombre y mediana: usar mediana
# aunque un link tenga mediana por name-match, si no tiene highway válido no se le deben propagar atributos
mask_usar_mediana = (
    mask_recibir_atts
    & links_fill["cap_med"].notna() #todos los que tienen medianas es porque si tienen nombre
    & links_fill["velprom_med"].notna()
    & links_fill["limvel_med"].notna()
)

links_fill.loc[mask_usar_mediana, "cap_final"] = links_fill.loc[mask_usar_mediana, "cap_med"]
links_fill.loc[mask_usar_mediana, "velprom_final"] = links_fill.loc[mask_usar_mediana, "velprom_med"]
links_fill.loc[mask_usar_mediana, "limvel_final"] = links_fill.loc[mask_usar_mediana, "limvel_med"]

# 2) Links sin nombre, pero con atributos originales y highway válido: conservar original
mask_sin_nombre_con_original = (
    mask_recibir_atts
    & links_fill[nombre].isna() #sin nombre
    & links_fill[capacidad].notna() #pero con atts heredados de transCAD
    & links_fill[vel_prom].notna()
    & links_fill[v0].notna()
)
links_fill.loc[mask_sin_nombre_con_original, "cap_final"] = links_fill.loc[mask_sin_nombre_con_original, capacidad]
links_fill.loc[mask_sin_nombre_con_original, "velprom_final"] = links_fill.loc[mask_sin_nombre_con_original, vel_prom]
links_fill.loc[mask_sin_nombre_con_original, "limvel_final"] = links_fill.loc[mask_sin_nombre_con_original, v0]

In [ ]:
print("Con mediana por nombre:")
print(links_fill["cap_med"].notna().sum())

print("Con final asignado:")
print(links_fill["cap_final"].notna().sum())

print("Tenían mediana pero NO pasaron a final por no ser highway válidos:")
perdidos = links_fill[
    links_fill["cap_med"].notna() &
    links_fill["cap_final"].isna()
]
print(len(perdidos))

print(perdidos["highway"].value_counts(dropna=False).head(30))

Con mediana por nombre:
32596
Con final asignado:
24163
Tenían mediana pero NO pasaron a final:
9352
highway
residential                         8556
living_street                        395
busway                               233
pedestrian                            82
track                                 34
path                                  20
footway                               14
['residential', 'path']               10
['residential', 'footway']             4
['pedestrian', 'steps']                2
['residential', 'living_street']       2
Name: count, dtype: int64


In [17]:
links_fill.to_file(os.path.join(shp_path, "links_atts_longitudinal.shp"))

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_2508/1844720284.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_fill.to_file(os.path.join(shp_path, "links_atts_longitudinal.shp"))
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'velprom_med' to 'velprom_me'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'velprom_final' to 'velprom_fi'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'limvel_final' to 'limvel_fin'
  ogr_write(


# EXPANSION A LO ANCHO

- los links que tenian atributo (match daniel) heredaron sus atributos a aquellos links con el mismo nombre __(expansion longitudinal)__
- faltaron cuchillas, retornornos, salidas que estaban sobre las mismas avenidas pero no pudieron heredar atributos __porque no tienen el mismo nombre__ (ya sea que tengan uno ligeramente diferente o que lo tengan NULL)
- __objetivo__: heredar a lo ancho para que las laterales, retornos y salidas de las avenidas tengan atts.

In [32]:
# 1) CRS a metros 
crs_original = links_fill.crs
links_m = links_fill.to_crs("EPSG:32613")

# 2) Links que ya tienen atributos finales (con nombre y sin nombre)
mask_final_atts = (
    links_fill["cap_final"].notna() &
    links_fill["velprom_final"].notna() &
    links_fill["limvel_final"].notna()
)
links_finalAtts = links_m.loc[mask_final_atts].copy()
print(f"Number of links with final attributes: {len(links_finalAtts):,}")

# 3) Links destino
highways_invalidos_buffer = [
    #"residential",
    "cycleway",
    "footway",
    "pedestrian",
    "path",
    "steps",
]
mask_highways_invalidos_buffer = (
    links_m["highway"].notna() # donde highway no es NULL (quitar lineas del metro)
    & ~links_m["highway"]
        .astype(str)
    .   str.contains("|".join(highways_invalidos_buffer), case=False, na=False)
)
mask_falta_atts = (
    links_m["cap_final"].isna() # y donde falta heredar
    & links_m["velprom_final"].isna()
    & links_m["limvel_final"].isna()
)
mask_destino = mask_highways_invalidos_buffer & mask_falta_atts
links_destino = links_m.loc[mask_destino].copy()

Number of links with final attributes: 24,163


In [34]:
# 4) Buffer alrededor de los links con atributos finales
buffer_m = 5
buffers_fuente = links_finalAtts[
    ["cap_final", "velprom_final", "limvel_final", "geometry"]
].copy()

buffers_fuente["source_idx"] = buffers_fuente.index
buffers_fuente["geometry"] = buffers_fuente.geometry.buffer(buffer_m)

In [36]:
buffers_fuente.to_file(os.path.join(shp_path, "buffers_fuente.shp"))

/var/folders/8s/00_wwq9j23b09mnd7gp105m00000gp/T/ipykernel_2508/293439200.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  buffers_fuente.to_file(os.path.join(shp_path, "buffers_fuente.shp"))
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'velprom_final' to 'velprom_fi'
  ogr_write(
/opt/anaconda3/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'limvel_final' to 'limvel_fin'
  ogr_write(
